# Decision Tree
Decision Tree implementation using CART algorithm

In [27]:
%pip install scikit-learn
%pip install biopython

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [28]:
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import GroupShuffleSplit
from Bio.Align import substitution_matrices
from collections import Counter
from treenode import TreeNode

In [29]:
#Importing Data
df = pd.read_csv("../data/processed_training_data_with_features.csv")
df.head(5)

,index,peptide,HLA,Label,hla_sequence,Peptide_AF1,Peptide_AF2,Peptide_AF3,Peptide_AF4,Peptide_AF5,...,HLA_Z5,HLA_boman,HLA_hydrophobicity,HLA_charge,HLA_molecular_weight,HLA_aliphatic_index,HLA_instability_index,HLA_isoelectric_point,HLA_mz,HLA_structural_class
0,0,KEHVFFSEY,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.173955,-0.346010,0.376536,-0.138244,-0.186528,...,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906,alpha+beta
1,1,DEGATLYRF,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.130313,-0.141878,0.624118,0.427633,0.340208,...,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906,alpha+beta
2,2,TLAARIKFL,HLA-A*02:01,0,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,-0.236566,-0.667240,0.421759,0.748899,0.552450,...,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991,alpha+beta
3,3,KETLNEYKQL,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.644379,-0.419675,0.461561,0.160180,0.170256,...,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906,alpha+beta
4,4,STTDAEACY,HLA-A*01:01,0,YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY,-0.016626,-0.045450,-0.193642,0.402509,-0.348259,...,-0.141471,1.769118,-0.447059,0.085599,4259.78264,63.235294,-7.820588,7.501996,2129.501106,alpha+beta


In [30]:
#Drop non feature columns
DROP_COLS = ["peptide", "hla_sequence", "index"]
TARGET_COL = "Label"

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

#Identify categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET_COL]
print(f"Encoding categorical columns: {cat_cols}")

#Hot encode to get numeric input for Decision Tree
df = pd.get_dummies(df, columns=cat_cols) 

# Split x and y values
y = df[TARGET_COL].values
X = df.drop(columns=[TARGET_COL])
print(df.columns.tolist()) #check what columns exist

print(f"Features: {X.shape}, Classes: {dict(zip(*np.unique(y, return_counts=True)))}")

Encoding categorical columns: ['HLA', 'Peptide_structural_class', 'HLA_structural_class']
['Label', 'Peptide_AF1', 'Peptide_AF2', 'Peptide_AF3', 'Peptide_AF4', 'Peptide_AF5', 'Peptide_BLOSUM1', 'Peptide_BLOSUM2', 'Peptide_BLOSUM3', 'Peptide_BLOSUM4', 'Peptide_BLOSUM5', 'Peptide_BLOSUM6', 'Peptide_BLOSUM7', 'Peptide_BLOSUM8', 'Peptide_BLOSUM9', 'Peptide_BLOSUM10', 'Peptide_PP1', 'Peptide_PP2', 'Peptide_PP3', 'Peptide_F1', 'Peptide_F2', 'Peptide_F3', 'Peptide_F4', 'Peptide_F5', 'Peptide_F6', 'Peptide_KF1', 'Peptide_KF2', 'Peptide_KF3', 'Peptide_KF4', 'Peptide_KF5', 'Peptide_KF6', 'Peptide_KF7', 'Peptide_KF8', 'Peptide_KF9', 'Peptide_KF10', 'Peptide_MSWHIM1', 'Peptide_MSWHIM2', 'Peptide_MSWHIM3', 'Peptide_E1', 'Peptide_E2', 'Peptide_E3', 'Peptide_E4', 'Peptide_E5', 'Peptide_PD1', 'Peptide_PD2', 'Peptide_PRIN1', 'Peptide_PRIN2', 'Peptide_PRIN3', 'Peptide_ProtFP1', 'Peptide_ProtFP2', 'Peptide_ProtFP3', 'Peptide_ProtFP4', 'Peptide_ProtFP5', 'Peptide_ProtFP6', 'Peptide_ProtFP7', 'Peptide_Prot

/var/folders/zp/5ww68fzx2x5dvm0pzf3hdprm0000gn/T/ipykernel_90328/2824346723.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


In [32]:
#Decision Tree implementation is based on the CART Algorithm
#Which uses Gini impurity as a metric 
# convert X to numpy array
feature_names = X.columns.tolist()
X = X.values  

def gini(y):
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    probs  = counts / len(y)
    return 1.0 - np.sum(probs ** 2)

#Information_gain calculates Gini-based information gain by subtracting weighted Gini impurity of the child from Gini impurity of the parent 
def information_gain(y_parent, y_left, y_right):
    n = len(y_parent)
    weighted_child = (len(y_left) / n)  * gini(y_left) + \
                     (len(y_right) / n) * gini(y_right)
    return gini(y_parent) - weighted_child

#Best_split scans all features and returns feature ID, threshold, and information gain for the split with highest information gain 
def best_split(X, y):
    best_gain = -1
    best_feat, best_thresh = None, None

    for feat in range(X.shape[1]):
        thresholds = np.unique(X[:, feat])

        for thresh in thresholds:
            left_mask  = X[:, feat] <= thresh
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            gain = information_gain(y, y[left_mask], y[right_mask])

            if gain > best_gain:
                best_gain   = gain
                best_feat   = feat
                best_thresh = thresh

    return best_feat, best_thresh, best_gain

#Build leaf 
def make_leaf(y, n_samples):
    unique, counts = np.unique(y, return_counts=True)
    label_counts = dict(zip(unique.tolist(), counts.tolist()))
    probs = {k: v / n_samples for k, v in label_counts.items()}

    return TreeNode(
        n_samples=n_samples,
        prediction_probs=probs,
        label_counts=label_counts
    )


#Build the tree recursively 
def build_tree(X, y, depth=0, max_depth=10, min_samples_split=2, min_samples_leaf=1):
    n_samples = len(y)
    
    if (
        depth >= max_depth
        or n_samples < min_samples_split
        or len(set(y)) == 1
    ):
        return make_leaf(y, n_samples)

    feat, thresh, gain = best_split(X, y)

    if feat is None or gain <= 0:
        return make_leaf(y, n_samples)

    left_mask = X[:, feat] <= thresh
    right_mask = ~left_mask

    if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
        return make_leaf(y, n_samples)

    # create node
    node = TreeNode(
        feature_idx=feat,
        feature_val=thresh,
        information_gain=gain,
        n_samples=n_samples
    )

    # recursion 
    node.left = build_tree(
        X[left_mask], y[left_mask],
        depth + 1, max_depth, min_samples_split, min_samples_leaf
    )

    node.right = build_tree(
        X[right_mask], y[right_mask],
        depth + 1, max_depth, min_samples_split, min_samples_leaf
    )

    return node

#Prediction function 
def predict_one(node, x):
    if node.is_leaf:
        return max(node.prediction_probs, key=node.prediction_probs.get)

    if x[node.feature_idx] <= node.feature_val:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)


def predict(root, X):
    return np.array([predict_one(root, x) for x in X])


#Feature importance aggregation
def get_feature_importances(node, n_features):

    importances = np.zeros(n_features)

    def walk(n):
        if n is None or n.is_leaf:
            return

        importances[n.feature_idx] += n.feature_importance

        walk(n.left)
        walk(n.right)

    walk(node)

    total = importances.sum()
    return importances / total if total > 0 else importances

#CARTClassifier wrapper function 
class CARTClassifier:
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf  = min_samples_leaf
        self.root              = None
        self.feature_names_    = None

    def fit(self, X, y, feature_names=None):
        self.root           = build_tree(X, y,
                                         max_depth=self.max_depth,
                                         min_samples_split=self.min_samples_split,
                                         min_samples_leaf=self.min_samples_leaf)
        self.feature_names_ = feature_names
        return self

    def predict(self, X):
        return predict(self.root, X)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

    def feature_importances(self):
        imps = get_feature_importances(self.root, len(self.feature_names_ or []))
        if self.feature_names_:
            return pd.Series(imps, index=self.feature_names_).sort_values(ascending=False)
        return imps
    

#Training and evaluating the model 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cart = CARTClassifier(max_depth=6, min_samples_split=5, min_samples_leaf=2)
cart.fit(X_train, y_train, feature_names=feature_names)

print(f"Train accuracy: {cart.score(X_train, y_train):.4f}")
print(f"Test  accuracy: {cart.score(X_test,  y_test):.4f}")

print("\nTop 10 most important features:")
print(cart.feature_importances().head(10))

Train accuracy: 0.7907
Test  accuracy: 0.7542

Top 10 most important features:
HLA_F3             0.533590
HLA_VSTPV2         0.112901
HLA_BLOSUM6        0.096553
HLA_PD1            0.042900
Peptide_BLOSUM1    0.034247
HLA_F4             0.022986
Peptide_VSTPV6     0.015372
HLA_E5             0.015123
Peptide_SVGER10    0.012287
Peptide_SVGER2     0.009327
dtype: float64
